# ゼロから作る Deep Learning ❸ 輪読会
## 第4ステージ「ニューラルネットワークを作る」 ― ステップ 42 〜 46

### これまでの内容

ステップ 37〜41 では、DeZero をテンソルに対応させ、`reshape`、`sum`、ブロードキャスト、行列積、アフィン変換を実装した。これにより、全結合層の計算

$$y = xW + b$$

と、その逆伝播ができるようになった。

今回は、その部品を使ってニューラルネットワークの**学習の仕組み**を組み立てる。前回の最後で試した線形回帰を正式に整理したあと、非線形なデータへ進み、繰り返し現れる処理を少しずつクラスへまとめていく。

### このノートブックの目標

| ステップ | テーマ | 到達点 |
|---|---|---|
| 42 | 線形回帰 | `予測 → 損失 → 逆伝播 → 更新` という学習ループを完成させる |
| 43 | ニューラルネットワーク | 活性化関数を挟んだ 2 層ネットワークで非線形なデータを学習する |
| 44 | パラメータをまとめるレイヤ | `Parameter`、`Layer`、`Linear` でパラメータ管理を自動化する |
| 45 | レイヤをまとめるレイヤ | `Model` と `MLP` でネットワーク全体をひとつのオブジェクトにする |
| 46 | Optimizer | パラメータ更新を `SGD` に任せ、学習ループを整理する |

最後には、学習コードを次の共通形にできる。

```python
y_pred = model(x)
loss = loss_function(y, y_pred)
model.cleargrads()
loss.backward()
optimizer.update()
```

## 準備：DeZero パッケージを読み込む

ステップ 41 までに作った自動微分とテンソル演算は、リポジトリの `dezero` パッケージから利用する。このリポジトリ自体はステップ 60 まで完成した状態だが、このノートブックでは各ステップで必要になった機能だけを順番に使う。

Jupyter の作業ディレクトリがリポジトリ直下でも `notebooks/` でも動くように、現在位置から上へたどって `dezero/` を探す。

In [ ]:
import sys
import weakref
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


current_dir = Path.cwd().resolve()
repo_root = next(
    (path for path in (current_dir, *current_dir.parents)
     if (path / "dezero").is_dir()),
    None,
)
if repo_root is None:
    raise RuntimeError("dezero ディレクトリを含むリポジトリが見つかりません")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dezero import Variable, no_grad
import dezero.functions as F


np.set_printoptions(precision=4, suppress=True)
print("repository:", repo_root)
print("NumPy version:", np.__version__)

---
# ステップ 42：線形回帰

## 42.1 トイ・データセット

まず、入力 $x$ と正解 $y$ の組を 100 個作る。正解はおおよそ $y=5+2x$ に従い、そこへ一様乱数のノイズを加える。

実際の学習ではファイルなどからデータを受け取るが、小さな人工データなら学習の仕組みそのものに集中できる。ここではデータを NumPy 配列のまま渡す。DeZero の関数が必要な時点で自動的に `Variable` へ変換するため、学習結果は明示的に包む場合と同じである。

In [ ]:
def make_linear_dataset(seed=0, size=100):
    rng = np.random.RandomState(seed)
    x = rng.rand(size, 1)
    y = 5 + 2 * x + rng.rand(size, 1)
    return x, y


linear_x, linear_y = make_linear_dataset()
print("x.shape:", linear_x.shape, "y.shape:", linear_y.shape)
print("先頭の3組:")
print(np.hstack([linear_x[:3], linear_y[:3]]))

## 42.2 線形回帰の理論

モデルは 1 入力・1 出力のアフィン変換とする。

$$\hat{y}=xW+b$$

$W$ は直線の傾き、$b$ は切片である。予測 $\hat{y}$ と正解 $y$ のずれは平均二乗誤差 (MSE) で測る。

$$L=\frac{1}{N}\sum_{i=1}^{N}(y_i-\hat{y}_i)^2$$

損失 $L$ が小さくなる方向は `backward()` で求め、学習率 $\eta$ を使って各パラメータを更新する。

$$W \leftarrow W-\eta\frac{\partial L}{\partial W},\qquad
b \leftarrow b-\eta\frac{\partial L}{\partial b}$$

## 42.3 線形回帰の実装

`predict_linear` がモデル、`mean_squared_error_simple` が損失関数に対応する。更新時は `.data` を直接変更する。パラメータ更新そのものを次の計算グラフへ含める必要がないためである。

In [ ]:
linear_W = Variable(np.zeros((1, 1)), name="W")
linear_b = Variable(np.zeros(1), name="b")


def predict_linear(x):
    return F.matmul(x, linear_W) + linear_b


def mean_squared_error_simple(x0, x1):
    diff = x0 - x1
    return F.sum(diff ** 2) / len(diff)


first_prediction = predict_linear(linear_x)
first_loss = mean_squared_error_simple(linear_y, first_prediction)
print("初期予測の形状:", first_prediction.shape)
print("初期 loss:", float(first_loss.data))

学習の 1 反復は次の 4 段階である。

1. モデルで予測する。
2. 損失を計算する。
3. 古い勾配を消してから逆伝播する。
4. 勾配の反対方向へパラメータを更新する。

`cleargrad()` を忘れると、以前の反復の勾配へ新しい勾配が加算されてしまう点に注意する。

In [ ]:
learning_rate_42 = 0.1
iterations_42 = 100
loss_history_42 = []

for iteration in range(iterations_42):
    y_pred = predict_linear(linear_x)
    loss = mean_squared_error_simple(linear_y, y_pred)

    linear_W.cleargrad()
    linear_b.cleargrad()
    loss.backward()

    # 他のステップと同様、更新直前の同じ時点にある loss, W, b を記録する。
    loss_history_42.append(float(loss.data))

    if iteration % 10 == 0 or iteration == iterations_42 - 1:
        print(
            f"iter {iteration:3d}: loss={float(loss.data):.4f}, "
            f"W={float(linear_W.data[0, 0]):.4f}, "
            f"b={float(linear_b.data[0]):.4f}"
        )

    linear_W.data -= learning_rate_42 * linear_W.grad.data
    linear_b.data -= learning_rate_42 * linear_b.grad.data

with no_grad():
    final_loss_42 = mean_squared_error_simple(linear_y, predict_linear(linear_x))
print(
    f"更新後: loss={float(final_loss_42.data):.4f}, "
    f"W={float(linear_W.data[0, 0]):.4f}, b={float(linear_b.data[0]):.4f}"
)

一様ノイズの平均は約 0.5 なので、学習後の直線は元の $5+2x$ より切片が約 0.5 大きくなる。データと学習結果、損失の推移を確認する。

In [ ]:
line_x = np.linspace(0, 1, 100).reshape(-1, 1)
line_y = predict_linear(line_x).data

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(linear_x, linear_y, s=18, alpha=0.65, label="data")
axes[0].plot(line_x, line_y, color="tab:red", linewidth=2, label="prediction")
axes[0].set(xlabel="x", ylabel="y", title="Linear regression")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(loss_history_42, color="tab:blue")
axes[1].set(xlabel="iteration", ylabel="MSE", title="Loss curve")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 42.4 【補足】DeZero の `mean_squared_error` 関数

上では式の構造が見えるように MSE を基本演算から作った。DeZero には同じ計算を専用の `Function` としてまとめた `F.mean_squared_error` も用意されている。専用版は中間変数を減らせるため、実際の学習ではこちらを使う。

今回は出力が 1 次元なので、`len(diff)`（バッチサイズ）で割る実装と全要素の平均は一致する。出力が多次元の場合は「何を平均する損失か」を意識する必要がある。

In [ ]:
current_prediction = predict_linear(linear_x)
simple_mse = mean_squared_error_simple(linear_y, current_prediction)
function_mse = F.mean_squared_error(linear_y, current_prediction)

print("基本演算で作った MSE:", float(simple_mse.data))
print("専用 Function の MSE:", float(function_mse.data))
np.testing.assert_allclose(simple_mse.data, function_mse.data)

> **ステップ 42 のまとめ**
>
> - 線形モデル $\hat{y}=xW+b$ と平均二乗誤差を組み合わせた。
> - 学習は `予測 → 損失 → 勾配初期化 → 逆伝播 → 更新` の繰り返しである。
> - `backward()` が勾配を求め、人が書くのはパラメータ更新の規則だけでよい。
> - 線形モデルが表現できるのは直線に限られる。

---
# ステップ 43：ニューラルネットワーク

## 43.1 `linear` 関数

ステップ 42 の `matmul(x, W) + b` は、DeZero の `F.linear(x, W, b)` でひとまとまりに書ける。

ただし、`linear` を何段重ねても、途中に非線形な処理がなければ全体は結局 1 本の直線と同じである。曲線を学ぶには**活性化関数**が必要になる。

## 43.2 非線形なデータセット

次は正弦波に一様分布 $U(0,1)$ のノイズを加えたデータを使う。ノイズの平均が 0.5 なので、点群の中心は純粋な正弦波より約 0.5 上に位置する。

$$y=\sin(2\pi x)+\varepsilon$$

この関係は 1 本の直線では表現できない。ニューラルネットワークが、データから滑らかな曲線を見つけられるかを試す。

In [ ]:
def make_sine_dataset(seed=0, size=100):
    # 各ステップの元コードと同じ乱数系列を再現する。
    np.random.seed(seed)
    x = np.random.rand(size, 1)
    y = np.sin(2 * np.pi * x) + np.random.rand(size, 1)
    return x, y


sine_x, sine_y = make_sine_dataset()

plt.figure(figsize=(6, 4))
plt.scatter(sine_x, sine_y, s=18, alpha=0.7)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Nonlinear toy dataset")
plt.grid(alpha=0.3)
plt.show()

## 43.3 活性化関数とニューラルネットワーク

ここではシグモイド関数を使う。

$$\sigma(x)=\frac{1}{1+e^{-x}}$$

シグモイドは受け取った中間値を 0〜1 の滑らかな非線形曲線へ変換する。全結合層の間へ挟むと、ネットワーク全体が曲線を表現できるようになる。最後の `Linear` にはシグモイドを適用しないため、最終出力 $\hat{y}$ 自体は 0〜1 に制限されず、負の値も表現できる。

$$x \xrightarrow{\text{Linear}(1,10)} h
\xrightarrow{\text{Sigmoid}} a
\xrightarrow{\text{Linear}(10,1)} \hat{y}$$

In [ ]:
def sigmoid_simple(x):
    return 1 / (1 + F.exp(-x))


sigmoid_input = Variable(np.array([-2.0, 0.0, 2.0]))
sigmoid_output = sigmoid_simple(sigmoid_input)
print("input :", sigmoid_input.data)
print("sigmoid:", sigmoid_output.data)

## 43.4 ニューラルネットワークの実装

まずは 4 個のパラメータ $W_1,b_1,W_2,b_2$ を手作業で用意する。隠れ層の幅 $H=10$ は、途中で 10 個の特徴を作るという意味である。

実際の学習には、同じ値をより効率よく計算する専用実装 `F.sigmoid` を使う。

In [ ]:
# このセルだけを再実行しても元コードと同じ乱数系列になるよう、
# データ生成から乱数状態をそろえる。
sine_x, sine_y = make_sine_dataset()

input_size, hidden_size, output_size = 1, 10, 1

manual_W1 = Variable(0.01 * np.random.randn(input_size, hidden_size), name="W1")
manual_b1 = Variable(np.zeros(hidden_size), name="b1")
manual_W2 = Variable(0.01 * np.random.randn(hidden_size, output_size), name="W2")
manual_b2 = Variable(np.zeros(output_size), name="b2")
manual_parameters = [manual_W1, manual_b1, manual_W2, manual_b2]


def predict_manual_network(x):
    h = F.linear(x, manual_W1, manual_b1)
    h = F.sigmoid(h)
    return F.linear(h, manual_W2, manual_b2)


shape_check = predict_manual_network(sine_x[:5])
print("入力:", sine_x[:5].shape, "→ 出力:", shape_check.shape)

In [ ]:
learning_rate_43 = 0.2
iterations_43 = 10_000
loss_history_43 = []

for iteration in range(iterations_43):
    y_pred = predict_manual_network(sine_x)
    loss = F.mean_squared_error(sine_y, y_pred)

    for parameter in manual_parameters:
        parameter.cleargrad()
    loss.backward()

    for parameter in manual_parameters:
        parameter.data -= learning_rate_43 * parameter.grad.data
    loss_history_43.append(float(loss.data))

    if iteration % 1000 == 0 or iteration == iterations_43 - 1:
        print(f"iter {iteration:5d}: loss={float(loss.data):.6f}")

In [ ]:
curve_x = np.linspace(0, 1, 200).reshape(-1, 1)
curve_y_43 = predict_manual_network(curve_x).data

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(sine_x, sine_y, s=18, alpha=0.6, label="data")
axes[0].plot(curve_x, curve_y_43, color="tab:red", linewidth=2, label="network")
axes[0].set(xlabel="x", ylabel="y", title="Two-layer neural network")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(loss_history_43)
axes[1].set(xlabel="iteration", ylabel="MSE", title="Loss curve")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

> **ステップ 43 のまとめ**
>
> - 全結合層の間に非線形な活性化関数を挟むと、曲線を表現できる。
> - 2 層ネットワークで正弦波状のデータへフィットできた。
> - パラメータが増えるたびに、初期化・勾配消去・更新のコードも増える。
> - 次は、関連するパラメータを `Layer` にまとめる。

---
# ステップ 44：パラメータをまとめるレイヤ

## 44.1 `Parameter` クラス

ニューラルネットワークには「入力や途中結果として使う変数」と「学習によって更新する変数」がある。どちらも自動微分の対象だが、後者だけを自動収集するため、`Variable` の目印となるサブクラスを作る。

`Parameter` 自体に新しい計算機能はない。**更新対象であることを型で表す**のが役割である。

In [ ]:
class Parameter(Variable):
    pass


sample_parameter = Parameter(np.array(1.0), name="sample")
print("Variable か:", isinstance(sample_parameter, Variable))
print("Parameter か:", isinstance(sample_parameter, Parameter))

## 44.2 `Layer` クラス

この段階の `Layer` は、属性へ代入された `Parameter` の名前を `_params` に記録する。Python では属性代入のたびに `__setattr__` が呼ばれるので、利用者が登録処理を別に書く必要はない。子 `Layer` の登録と再帰的な探索はステップ 45 で追加する。

主な役割は次の 3 つである。

- `__call__`：`forward` を呼び、入出力を弱参照で保持する。
- `params`：自分に登録されたパラメータを順に返す。
- `cleargrads`：登録済みパラメータの勾配をまとめて消す。

弱参照を使うのは、レイヤが過去の入出力を強く保持して不要なメモリを残さないためである。

In [ ]:
class Layer:
    def __init__(self):
        self._params = set()

    def __setattr__(self, name, value):
        if isinstance(value, Parameter):
            self._params.add(name)
        super().__setattr__(name, value)

    def __call__(self, *inputs):
        outputs = self.forward(*inputs)
        if not isinstance(outputs, tuple):
            outputs = (outputs,)
        self.inputs = [weakref.ref(x) for x in inputs]
        self.outputs = [weakref.ref(y) for y in outputs]
        return outputs if len(outputs) > 1 else outputs[0]

    def forward(self, *inputs):
        raise NotImplementedError()

    def params(self):
        for name in sorted(self._params):
            yield self.__dict__[name]

    def cleargrads(self):
        for parameter in self.params():
            parameter.cleargrad()

## 44.3 `Linear` レイヤ

`Linear(out_size)` は出力サイズだけを受け取る。入力サイズは、最初のデータが流れたときに `x.shape[1]` から判断して重みを初期化する。この仕組みを**遅延初期化**という。

重みは入力数 $I$ に応じて $\sqrt{1/I}$ 倍する。層を重ねたときに値が極端に大きくなったり小さくなったりするのを抑えるためである。

In [ ]:
class Linear(Layer):
    def __init__(self, out_size, nobias=False, dtype=np.float32, in_size=None):
        super().__init__()
        self.in_size = in_size
        self.out_size = out_size
        self.dtype = dtype

        self.W = Parameter(None, name="W")
        if self.in_size is not None:
            self._init_W()

        self.b = None if nobias else Parameter(
            np.zeros(out_size, dtype=dtype), name="b"
        )

    def _init_W(self):
        input_size, output_size = self.in_size, self.out_size
        weight = np.random.randn(input_size, output_size).astype(self.dtype)
        weight *= np.sqrt(1 / input_size)
        self.W.data = weight

    def forward(self, x):
        if self.W.data is None:
            self.in_size = x.shape[1]
            self._init_W()
        return F.linear(x, self.W, self.b)


demo_linear = Linear(3)
print("最初の W:", demo_linear.W.data)
demo_output = demo_linear(np.ones((2, 4)))
print("入力を流した後の W.shape:", demo_linear.W.shape)
print("出力 shape:", demo_output.shape)
print("登録パラメータ:", [parameter.name for parameter in demo_linear.params()])

## 44.4 `Layer` を使ったニューラルネットワーク

ステップ 43 と同じネットワークを 2 個の `Linear` レイヤで作る。重み・バイアスの生成はレイヤへ移り、学習側は各レイヤの `params()` をたどるだけになる。

In [ ]:
# Step 44 の独立したスクリプトと同じ乱数系列から始める。
sine_x, sine_y = make_sine_dataset()
layer1_44 = Linear(10)
layer2_44 = Linear(1)


def predict_with_layers(x):
    h = F.sigmoid(layer1_44(x))
    return layer2_44(h)


learning_rate_44 = 0.2
iterations_44 = 10_000
loss_history_44 = []

for iteration in range(iterations_44):
    y_pred = predict_with_layers(sine_x)
    loss = F.mean_squared_error(sine_y, y_pred)

    layer1_44.cleargrads()
    layer2_44.cleargrads()
    loss.backward()

    for layer in (layer1_44, layer2_44):
        for parameter in layer.params():
            parameter.data -= learning_rate_44 * parameter.grad.data
    loss_history_44.append(float(loss.data))

    if iteration % 1000 == 0 or iteration == iterations_44 - 1:
        print(f"iter {iteration:5d}: loss={float(loss.data):.6f}")

> **ステップ 44 のまとめ**
>
> - `Parameter` は学習対象を見分けるための `Variable` である。
> - `Layer` はパラメータの収集と勾配の初期化を担当する。
> - `Linear` は重み・バイアスとアフィン変換をひとまとめにする。
> - 遅延初期化により、利用者は入力サイズを省略できる。
> - まだ 2 個のレイヤを学習側で個別に扱っているため、次はネットワーク全体をまとめる。

---
# ステップ 45：レイヤをまとめるレイヤ

## 45.1 `Layer` クラスの拡張

ステップ 44 の `Layer` は `Parameter` だけを登録した。ここで `Layer` 自身も登録対象へ加え、`params()` が子レイヤを見つけたら再帰的に中へ入るように拡張する。

```text
TwoLayerNet
├── l1: Linear
│   ├── W
│   └── b
└── l2: Linear
    ├── W
    └── b
```

この階層構造により、ネットワークが深くなっても `model.params()` という同じ入口から全パラメータへアクセスできる。

In [ ]:
# Notebook 上で既存の Linear インスタンスとのクラス同一性を保つため、
# Step 44 の Layer にメソッドを追加する形で拡張する。
def _nested_layer_setattr(self, name, value):
    if isinstance(value, (Parameter, Layer)):
        self._params.add(name)
    object.__setattr__(self, name, value)


def _nested_layer_params(self):
    for name in sorted(self._params):
        obj = self.__dict__[name]
        if isinstance(obj, Layer):
            yield from obj.params()
        else:
            yield obj


Layer.__setattr__ = _nested_layer_setattr
Layer.params = _nested_layer_params

## 45.2 `Model` クラス

`Model` は `Layer` の一種であり、ネットワーク全体を表す意味上の基底クラスである。新しい微分規則は必要なく、部品であるレイヤを属性として持ち、`forward` で接続する。さらに `plot` を持たせると、モデルが作った計算グラフを可視化できる。画像への変換には外部コマンドの Graphviz が必要だが、DOT 形式のグラフ記述自体は DeZero だけで生成できる。

In [ ]:
from dezero import utils


class Model(Layer):
    def plot(self, *inputs, to_file="model.png"):
        output = self.forward(*inputs)
        return utils.plot_dot_graph(output, verbose=True, to_file=to_file)


class TwoLayerNet(Model):
    def __init__(self, hidden_size, out_size):
        super().__init__()
        self.l1 = Linear(hidden_size)
        self.l2 = Linear(out_size)

    def forward(self, x):
        h = F.sigmoid(self.l1(x))
        return self.l2(h)


sine_x, sine_y = make_sine_dataset()
model_45 = TwoLayerNet(hidden_size=10, out_size=1)
_ = model_45(sine_x[:1])  # 遅延初期化を完了させる

print("モデルの出力 shape:", model_45(sine_x[:5]).shape)
print("モデルが集めたパラメータ:")
for parameter in model_45.params():
    print(f"  {parameter.name}: shape={parameter.shape}")

dot_source = utils.get_dot_graph(model_45(sine_x[:1]), verbose=True)
print("計算グラフの DOT 行数:", len(dot_source.splitlines()))

## 45.3 `Model` を使って問題を解く

`model.cleargrads()` は内部の全レイヤへ再帰的に届く。更新時も `model.params()` だけを見ればよく、ネットワークの具体的な層数が学習ループから消える。

In [ ]:
# 上の形状確認でも重みを初期化したため、同じ初期値から学習できるよう作り直す
sine_x, sine_y = make_sine_dataset()
model_45 = TwoLayerNet(hidden_size=10, out_size=1)

learning_rate_45 = 0.2
iterations_45 = 10_000
loss_history_45 = []

for iteration in range(iterations_45):
    y_pred = model_45(sine_x)
    loss = F.mean_squared_error(sine_y, y_pred)

    model_45.cleargrads()
    loss.backward()

    for parameter in model_45.params():
        parameter.data -= learning_rate_45 * parameter.grad.data
    loss_history_45.append(float(loss.data))

    if iteration % 1000 == 0 or iteration == iterations_45 - 1:
        print(f"iter {iteration:5d}: loss={float(loss.data):.6f}")

## 45.4 `MLP` クラス

`TwoLayerNet` を一般化し、各全結合層の出力サイズをタプルで受け取るようにすれば、多層パーセプトロン (Multi-Layer Perceptron; MLP) を作れる。

たとえば `MLP((10, 1))` は「隠れ層 10、出力層 1」、`MLP((20, 10, 3))` は「20 → 10 → 3」の 3 層を表す。最後の層以外に活性化関数を適用する。各レイヤは `setattr` でモデルへ登録し、同時に `layers` リストへも入れる。前者はパラメータの再帰収集、後者は順伝播の順序保持に使う。

In [ ]:
class MLP(Model):
    def __init__(self, output_sizes, activation=F.sigmoid):
        super().__init__()
        self.activation = activation
        self.layers = []

        for index, out_size in enumerate(output_sizes):
            layer = Linear(out_size)
            setattr(self, f"l{index}", layer)
            self.layers.append(layer)

    def forward(self, x):
        for layer in self.layers[:-1]:
            x = self.activation(layer(x))
        return self.layers[-1](x)


np.random.seed(0)
demo_mlp = MLP((20, 10, 3))
demo_score = demo_mlp(np.zeros((4, 2)))
print("MLP((20, 10, 3)) の出力 shape:", demo_score.shape)
demo_parameters = list(demo_mlp.params())
print("Parameter テンソル数:", len(demo_parameters))
print("学習可能なスカラー数:", sum(parameter.size for parameter in demo_parameters))

> **ステップ 45 のまとめ**
>
> - レイヤを子レイヤとして登録し、パラメータを再帰的に集められるようにした。
> - `Model` はネットワーク全体を 1 個の `Layer` として扱う。
> - `cleargrads()` と `params()` の呼び出し先がモデル 1 個にまとまった。
> - `MLP` は出力サイズの並びから任意の深さの全結合ネットワークを作る。
> - パラメータ更新式はまだ学習ループ内に残っている。

---
# ステップ 46：Optimizer によるパラメータ更新

## 46.1 `Optimizer` クラス

最適化手法は、勾配を使ってパラメータをどう更新するかを決める。基底クラス `Optimizer` は次の流れを共通化する。

1. `setup(model)` で更新対象のモデルを登録する。
2. `update()` で勾配を持つパラメータを集める。
3. 各パラメータに、手法固有の `update_one()` を適用する。

`Optimizer` は勾配を計算しない。勾配計算はこれまでどおり `loss.backward()` の役割である。

In [ ]:
class Optimizer:
    def __init__(self):
        self.target = None

    def setup(self, target):
        self.target = target
        return self

    def update(self):
        parameters = [
            parameter
            for parameter in self.target.params()
            if parameter.grad is not None
        ]
        for parameter in parameters:
            self.update_one(parameter)

    def update_one(self, parameter):
        raise NotImplementedError()

## 46.2 `SGD` クラス

確率的勾配降下法 (Stochastic Gradient Descent; SGD) の更新式は、ステップ 42 から使ってきたものと同じである。SGD の確率性は、ランダムに選んだミニバッチから勾配を求めることで生まれる。ここでは毎回 100 件すべてを使うため、厳密にはバッチ勾配降下法として動く。

$$\theta \leftarrow \theta-\eta\frac{\partial L}{\partial\theta}$$

モデルの種類やパラメータ名に依存しないため、実装は 1 行で済む。

In [ ]:
class SGD(Optimizer):
    def __init__(self, learning_rate=0.01):
        super().__init__()
        self.learning_rate = learning_rate

    def update_one(self, parameter):
        parameter.data -= self.learning_rate * parameter.grad.data

## 46.3 `SGD` を使って問題を解く

`MLP((10, 1))` と `SGD(0.2)` を組み合わせる。学習ループから具体的な更新式が消え、モデルの内部構造にも最適化手法にも依存しない形になった。

In [ ]:
sine_x, sine_y = make_sine_dataset()
model_46 = MLP((10, 1))
optimizer_46 = SGD(learning_rate=0.2).setup(model_46)

iterations_46 = 10_000
loss_history_46 = []

for iteration in range(iterations_46):
    y_pred = model_46(sine_x)
    loss = F.mean_squared_error(sine_y, y_pred)

    model_46.cleargrads()
    loss.backward()
    optimizer_46.update()
    loss_history_46.append(float(loss.data))

    if iteration % 1000 == 0 or iteration == iterations_46 - 1:
        print(f"iter {iteration:5d}: loss={float(loss.data):.6f}")

`Layer`、`Model`、`Optimizer` は計算結果を変えるためではなく、責務を分けるための抽象化である。同じ乱数シード・同じネットワーク・同じ SGD を使ったステップ 44〜46 の損失が一致することを確かめる。

In [ ]:
np.testing.assert_allclose(loss_history_44, loss_history_45, rtol=0, atol=0)
np.testing.assert_allclose(loss_history_45, loss_history_46, rtol=0, atol=0)

for step, history in (
    (43, loss_history_43),
    (44, loss_history_44),
    (45, loss_history_45),
    (46, loss_history_46),
):
    print(f"step {step}: final loss = {history[-1]:.6f}")

print("step 44〜46 の学習曲線は完全に一致しました")

In [ ]:
curve_y_46 = model_46(curve_x).data

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(sine_x, sine_y, s=18, alpha=0.6, label="data")
axes[0].plot(curve_x, curve_y_46, color="tab:red", linewidth=2, label="MLP")
axes[0].set(xlabel="x", ylabel="y", title="MLP trained with SGD")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(loss_history_46, color="tab:blue")
axes[1].set(xlabel="iteration", ylabel="MSE", title="Loss curve")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 46.4 SGD 以外の最適化手法

SGD は現在の勾配だけで更新方向を決める。MomentumSGD は「速度」$v$ を持ち、前回までの更新方向を少し残す。

$$v \leftarrow \mu v-\eta g,\qquad \theta \leftarrow \theta+v$$

$\mu$ はモーメンタム係数である。同じ `Optimizer` を継承し、`update_one` だけを交換すれば実装できる。

In [ ]:
class MomentumSGD(Optimizer):
    def __init__(self, learning_rate=0.01, momentum=0.9):
        super().__init__()
        self.learning_rate = learning_rate
        self.momentum = momentum
        self.velocities = {}

    def update_one(self, parameter):
        key = id(parameter)
        if key not in self.velocities:
            self.velocities[key] = np.zeros_like(parameter.data)

        velocity = self.velocities[key]
        velocity *= self.momentum
        velocity -= self.learning_rate * parameter.grad.data
        parameter.data += velocity


print("Optimizer を継承した手法:", SGD.__name__, MomentumSGD.__name__)

DeZero の `optimizers` モジュールには、ほかにも AdaGrad、AdaDelta、Adam などがある。学習ループを変えずに Optimizer だけを交換できることが、抽象化の利点である。

## 補足：DeZero パッケージ版との対応

ここまでノートブック内で作った `Linear`、`MLP`、`SGD` は、仕組みを見やすくした CPU 用の簡略版である。対応する実用版はリポジトリの `dezero.layers`、`dezero.models`、`dezero.optimizers` に整理され、後のステップで使う GPU 対応なども含む。実際のコードではそれらを import して使える。

次のセルでは、パッケージ版だけを使って学習の 1 反復が動くことを確認する。

In [ ]:
from dezero.models import MLP as DeZeroMLP
from dezero.optimizers import SGD as DeZeroSGD


np.random.seed(1)
package_model = DeZeroMLP((10, 1))
package_optimizer = DeZeroSGD(0.2).setup(package_model)

package_prediction = package_model(sine_x)
package_loss = F.mean_squared_error(sine_y, package_prediction)
package_model.cleargrads()
package_loss.backward()
package_optimizer.update()

package_parameters = list(package_model.params())
print("1 反復の loss:", float(package_loss.data))
print("パッケージ版が管理する Parameter テンソル数:", len(package_parameters))
print("学習可能なスカラー数:", sum(parameter.size for parameter in package_parameters))
assert len(package_parameters) == 4

### DeZero と PyTorch の対応

| 役割 | DeZero | PyTorch |
|---|---|---|
| 学習対象の変数 | `Parameter` | `torch.nn.Parameter` |
| レイヤの基底 | `Layer` | `torch.nn.Module` |
| 全結合層 | `Linear` | `torch.nn.Linear` |
| モデル | `Model` / `MLP` | `nn.Module` を継承したモデル |
| 勾配の初期化 | `model.cleargrads()` | `optimizer.zero_grad()` |
| 逆伝播 | `loss.backward()` | `loss.backward()` |
| 更新 | `optimizer.update()` | `optimizer.step()` |

名前や細部は異なるが、パラメータをレイヤに所属させ、モデルが再帰的に集め、Optimizer が更新するという設計は共通している。

> **ステップ 46 のまとめ**
>
> - `Optimizer` はモデルから更新対象を集め、更新処理を統一する。
> - `SGD` は勾配の反対方向へ学習率ぶん進む。
> - モデルの構造と最適化手法が分離され、どちらも交換しやすくなった。
> - 学習ループは `model`、`loss`、`optimizer` の役割が見える共通形になった。

---
# まとめ ― 第 4 ステージ中盤（ステップ 42〜46）

この 5 ステップでは、テンソル演算から出発して、ニューラルネットワークを学習させるための構造を作った。

| 段階 | 手作業で管理していたもの | 導入した仕組み |
|---|---|---|
| ステップ 42 | $W,b$ と更新式 | 線形回帰の学習ループ |
| ステップ 43 | $W_1,b_1,W_2,b_2$ | 活性化関数を持つ 2 層ネットワーク |
| ステップ 44 | 各パラメータの列挙 | `Parameter` / `Layer` / `Linear` |
| ステップ 45 | 各レイヤの列挙 | `Model` / `MLP` と再帰的なパラメータ収集 |
| ステップ 46 | パラメータ更新式 | `Optimizer` / `SGD` |

### 学習時の情報の流れ

```text
x ──→ model ──→ prediction ──→ loss
                                 │
                                 ▼ backward
                           parameter.grad
                                 │
                                 ▼ optimizer.update
                          parameter.data を更新
```

自動微分の計算能力そのものに加え、**どの値を学習し、どの単位でまとめ、誰が更新するか**が整理されたことで、小さな深層学習フレームワークらしい形になった。

### 次のステップ（47 以降）

次は回帰から分類へ進む。ステップ 47 で `softmax` と交差エントロピーを導入し、ステップ 48 では多クラス分類を学習する。その後、`Dataset` と `DataLoader` でデータの取り扱いも整理していく。